# Optimizers & Dropout — Theory to Practice

**Course:** ML, Deep Learning & Computer Vision — Phase 3  
**Prerequisites:** Phase 3 theory (neurons, backprop, loss)  
**Estimated time:** 2 hours (theory) + 1 hour (experiments)  

---

This notebook covers:

1. **The math** behind each optimiser — not just the update rule, but *why* it works
2. **Head-to-head comparison** on a 2D loss surface — see the difference
3. **MNIST experiments** — train the same network with SGD, Momentum, RMSProp, Adam
4. **Dropout** — theory, implementation from scratch, and MNIST ablation
5. **Practical guidelines** — what to use when

By the end, you'll understand exactly what `torch.optim.Adam(model.parameters(), lr=1e-3)` is doing under the hood.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

---
# Part 1 — The Theory Behind Each Optimiser

All optimisers follow the same skeleton:

```
for each training step:
    1. Compute gradient g = ∇L(w)
    2. Compute an update Δw from g (this is where they differ)
    3. Apply: w ← w − lr × Δw
```

The differences lie entirely in step 2. Each optimizer maintains different **state** between steps.

---

## 1.1 Vanilla SGD — the baseline

**Update rule:**
$$w_{t+1} = w_t - \eta \cdot g_t$$

where $g_t = \nabla_w \mathcal{L}(w_t)$ is the gradient at step $t$ and $\eta$ is the learning rate.

**State:** None. Each step only uses the current gradient.

**Problems:**
1. **Oscillates on elongated surfaces** — if the loss is steep in one direction and flat in another, SGD bounces back and forth across the steep direction
2. **Gets stuck at saddle points** — gradient ≈ 0, so updates are tiny
3. **Noisy** — each mini-batch gives a different gradient estimate; no smoothing

In [ ]:
# SGD from scratch
class VanillaSGD:
    """Stochastic Gradient Descent — the simplest optimizer."""
    
    def __init__(self, lr=0.01):
        self.lr = lr
    
    def step(self, params, grads):
        # That's it. Literally one line.
        return params - self.lr * grads

print("SGD: w ← w - η·g")
print("No memory. No state. Just follow the gradient.")
print("This is the baseline everything else improves upon.")

## 1.2 SGD with Momentum — rolling downhill

**Key idea:** Maintain a *velocity* vector that accumulates past gradients. Like a ball rolling downhill — it builds speed in consistent directions and dampens oscillations.

**Update rule:**
$$v_t = \beta \cdot v_{t-1} + g_t$$
$$w_{t+1} = w_t - \eta \cdot v_t$$

where $\beta$ (typically 0.9) is the momentum coefficient.

**State:** velocity $v$ (same shape as parameters).

**Why it works:**
- In the oscillating direction, gradients flip sign each step → they cancel out in the velocity
- In the consistent direction, gradients point the same way → velocity builds up
- At saddle points, the accumulated velocity carries the optimizer through

**Think of it as:** an exponentially weighted moving average of gradients, with decay rate $\beta$. The effective window is $\frac{1}{1-\beta}$ steps (so $\beta=0.9$ averages ~10 recent gradients).

In [ ]:
class MomentumSGD:
    """SGD with Momentum — adds a velocity term."""
    
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = 0  # velocity starts at zero
    
    def step(self, params, grads):
        # Accumulate velocity
        self.v = self.beta * self.v + grads
        #         ^^^^^^^^^^^^        ^^^^^
        #         keep 90% of         add current
        #         old velocity         gradient
        
        return params - self.lr * self.v

print("Momentum: v ← βv + g,  then  w ← w - η·v")
print(f"With β=0.9, velocity averages ~{1/(1-0.9):.0f} recent gradients.")
print(f"With β=0.99, velocity averages ~{1/(1-0.99):.0f} recent gradients.")
print("\nEffect: smooths out noise, accelerates in consistent directions.")

## 1.3 RMSProp — adaptive per-parameter learning rates

**Key insight:** Different parameters need different learning rates. A weight that always gets large gradients needs a *smaller* step. A weight with tiny gradients needs a *bigger* step.

**Update rule:**
$$s_t = \beta \cdot s_{t-1} + (1-\beta) \cdot g_t^2$$
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{s_t} + \epsilon} \cdot g_t$$

where $s_t$ is a running average of squared gradients (per parameter).

**State:** running squared gradient average $s$ (same shape as parameters).

**Why it works:**
- Dividing by $\sqrt{s_t}$ normalises the step size by recent gradient magnitude
- Parameters with consistently large gradients → large $s$ → *smaller* effective lr
- Parameters with consistently small gradients → small $s$ → *larger* effective lr
- This automatically handles the elongated valley problem — each dimension gets its own scale

**$\epsilon$ (typically 1e-7 or 1e-8):** prevents division by zero when $s_t \approx 0$.

In [ ]:
class RMSProp:
    """RMSProp — adaptive per-parameter learning rates."""
    
    def __init__(self, lr=0.001, beta=0.999, eps=1e-8):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.s = 0  # running average of squared gradients
    
    def step(self, params, grads):
        # Track squared gradient magnitude
        self.s = self.beta * self.s + (1 - self.beta) * grads**2
        #         ^^^^^^^^^^^^^^^     ^^^^^^^^^^^^^^^^^^^^^^^^^^
        #         keep 99.9% of old   add 0.1% of current grad²
        
        # Divide gradient by its running RMS → normalise step size
        return params - self.lr * grads / (np.sqrt(self.s) + self.eps)
        #                        ^^^^^    ^^^^^^^^^^^^^^^^^
        #                        raw      adaptive denominator:
        #                        gradient  large grads → small step
        #                                  small grads → large step

print("RMSProp: s ← βs + (1−β)g²,  then  w ← w − η·g/√(s+ε)")
print("\nPer-parameter adaptive lr. No momentum though — that's what Adam adds.")

## 1.4 Adam — the best of both worlds

Adam combines **momentum** (first moment) with **RMSProp** (second moment), plus **bias correction** for the initialisation.

**Update rule:**

$$m_t = \beta_1 \cdot m_{t-1} + (1-\beta_1) \cdot g_t \quad \text{(first moment — momentum)}$$
$$v_t = \beta_2 \cdot v_{t-1} + (1-\beta_2) \cdot g_t^2 \quad \text{(second moment — RMSProp)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \quad \text{(bias correction for } m \text{)}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{(bias correction for } v \text{)}$$
$$w_{t+1} = w_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

**State:** first moment $m$, second moment $v$, step counter $t$.

**Why bias correction?** Both $m$ and $v$ are initialised to 0. In early steps, they're biased toward 0 because they haven't seen enough gradients yet. Dividing by $(1-\beta^t)$ compensates — at step 1 with $\beta_1=0.9$, we divide by 0.1, effectively multiplying the first gradient by 10.

**Default hyperparameters** (Kingma & Ba, 2015):
- $\beta_1 = 0.9$ (momentum decay)
- $\beta_2 = 0.999$ (RMSProp decay)
- $\eta = 10^{-3}$ or $5 \times 10^{-4}$
- $\epsilon = 10^{-8}$

These rarely need changing.

In [ ]:
class AdamFromScratch:
    """Adam — Momentum + RMSProp + bias correction."""
    
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = 0       # first moment (momentum)
        self.v = 0       # second moment (RMSProp)
        self.t = 0       # step counter
    
    def step(self, params, grads):
        self.t += 1
        
        # First moment: exponential moving average of gradients (= momentum)
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        
        # Second moment: exponential moving average of squared gradients (= RMSProp)
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads**2
        
        # Bias correction: compensate for initialisation at zero
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        # Update: momentum direction, scaled by adaptive per-parameter lr
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Show the bias correction effect
print("=== Bias correction at early steps ===")
print("Without correction, m and v are biased toward 0 in early steps.")
print("\nStep  β₁ᵗ    1−β₁ᵗ   Correction factor (1/1−β₁ᵗ)")
for t in range(1, 8):
    b = 0.9 ** t
    print(f"  {t}    {b:.4f}  {1-b:.4f}   ×{1/(1-b):.2f}")
print("\nAt step 1: multiply by 10× to compensate.")
print("By step 10: correction is negligible (~1.05×).")

---
# Part 2 — Visual Comparison on a 2D Loss Surface

Let's run all four optimisers on the same elongated bowl and see the difference.

In [ ]:
# Loss surface: f(w1, w2) = 0.5*w1² + 10*w2²
# This is 20× steeper in the w2 direction → SGD will oscillate

def loss_fn(w):
    return 0.5 * w[0]**2 + 10 * w[1]**2

def grad_fn(w):
    return np.array([w[0], 20 * w[1]])

def run_optimizer(opt_class, start, n_steps=100, **kwargs):
    opt = opt_class(**kwargs)
    w = start.copy()
    path = [w.copy()]
    for _ in range(n_steps):
        g = grad_fn(w)
        w = opt.step(w, g)
        path.append(w.copy())
    return np.array(path)

start = np.array([4.0, 3.0])
configs = [
    ("SGD",      VanillaSGD,      {"lr": 0.04},              "#E24B4A", '--'),
    ("Momentum", MomentumSGD,     {"lr": 0.04, "beta": 0.9}, "#378ADD", '-'),
    ("RMSProp",  RMSProp,         {"lr": 0.3},               "#7F77DD", '-'),
    ("Adam",     AdamFromScratch, {"lr": 0.3},               "#1D9E75", '-'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
zz = 0.5 * xx**2 + 10 * yy**2

for ax, (name, cls, kw, color, ls) in zip(axes, configs):
    path = run_optimizer(cls, start, 100, **kw)
    ax.contour(xx, yy, zz, levels=25, cmap='Blues', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], 'o-', color=color, markersize=2, linewidth=1.2, alpha=0.8)
    ax.plot(0, 0, '*', color='red', markersize=12, zorder=5)
    ax.plot(start[0], start[1], 's', color=color, markersize=8)
    final_loss = loss_fn(path[-1])
    ax.set_title(f'{name}\nFinal loss: {final_loss:.4f}', fontweight='bold', color=color)
    ax.set_xlim(-5, 5); ax.set_ylim(-4, 4)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
    ax.grid(True, alpha=0.2)

plt.suptitle('Optimiser paths on an elongated bowl (condition ratio 20:1)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("SGD oscillates wildly. Momentum smooths the path.")
print("RMSProp adapts per-dimension. Adam combines both → fastest convergence.")

---
# Part 3 — MNIST Head-to-Head: Which Optimiser Wins?

We'll train the **exact same network** with 4 different optimisers and compare convergence speed and final accuracy.

### Setup

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
# Define the network — same architecture for all experiments
def make_model():
    """Fresh model instance (same weights via manual seed)."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Linear(128, 10),
    ).to(device)

# Training utilities
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

print("Model and utilities ready.")

In [ ]:
# === RUN ALL FOUR OPTIMISERS ===
N_EPOCHS = 10
criterion = nn.CrossEntropyLoss()

optimiser_configs = {
    "SGD (lr=0.01)": lambda m: torch.optim.SGD(m.parameters(), lr=0.01),
    "SGD+Momentum (lr=0.01)": lambda m: torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9),
    "RMSProp (lr=0.001)": lambda m: torch.optim.RMSprop(m.parameters(), lr=0.001),
    "Adam (lr=0.001)": lambda m: torch.optim.Adam(m.parameters(), lr=0.001),
}

results = {}
for name, opt_fn in optimiser_configs.items():
    print(f"\n{'='*60}")
    print(f"Training with {name}")
    print(f"{'='*60}")
    
    model = make_model()  # fresh model with same init
    optimizer = opt_fn(model)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()
    
    for epoch in range(1, N_EPOCHS + 1):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, test_loader, criterion)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        
        if epoch % 2 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d} | Train: {ta:.2%}, loss={tl:.4f} | Val: {va:.2%}, loss={vl:.4f}")
    
    elapsed = time.time() - t0
    results[name] = history
    print(f"  Done in {elapsed:.1f}s | Best val acc: {max(history['val_acc']):.2%}")

In [ ]:
# === PLOT COMPARISON ===
colors = {'SGD (lr=0.01)': '#E24B4A', 'SGD+Momentum (lr=0.01)': '#378ADD',
          'RMSProp (lr=0.001)': '#7F77DD', 'Adam (lr=0.001)': '#1D9E75'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, hist in results.items():
    c = colors[name]
    axes[0].plot(range(1, N_EPOCHS+1), hist['val_loss'], 'o-', color=c, linewidth=2,
                markersize=4, label=name)
    axes[1].plot(range(1, N_EPOCHS+1), hist['val_acc'], 'o-', color=c, linewidth=2,
                markersize=4, label=name)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Convergence speed (loss)', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Convergence speed (accuracy)', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.2)
axes[1].set_ylim(0.9, 1.0)

plt.suptitle('Optimiser comparison on MNIST (same model, same init)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\n=== FINAL RESULTS ===")
print(f"{'Optimiser':30s} {'Best Val Acc':>12s} {'Final Val Loss':>15s} {'Epoch 1 Acc':>12s}")
print('-' * 72)
for name, hist in results.items():
    print(f"{name:30s} {max(hist['val_acc']):12.2%} {hist['val_loss'][-1]:15.4f} {hist['val_acc'][0]:12.2%}")

### What to expect from the results

- **SGD** converges slowly — it may not even reach 95% in 10 epochs with lr=0.01
- **SGD + Momentum** is significantly faster — momentum smooths the updates
- **RMSProp** converges fast — adaptive lr handles different parameter scales
- **Adam** typically converges fastest in early epochs — it combines both benefits

**Important nuance:** With enough epochs and careful lr tuning, SGD+Momentum can sometimes *match or beat* Adam on final accuracy. Adam is faster to converge but can sometimes generalise slightly worse. This is an active research topic.

---
# Part 4 — Dropout: Theory & Practice

## 4.1 What is dropout?

During training, randomly set each neuron's output to 0 with probability $p$.

$$h_i^{\text{dropped}} = \begin{cases} 0 & \text{with probability } p \\ \frac{h_i}{1-p} & \text{with probability } 1-p \end{cases}$$

The $\frac{1}{1-p}$ scaling (called **inverted dropout**) ensures the expected value stays the same. This way, we don't need to change anything at test time.

## 4.2 Why does dropout work?

Three complementary explanations:

1. **Ensemble interpretation:** Each training step uses a *different sub-network* (a random subset of neurons). At test time, we use all neurons — which is like averaging the predictions of all possible sub-networks. Dropout ≈ training an exponential number of models for free.

2. **Redundancy argument:** No single neuron can be relied upon (it might be dropped). The network is forced to learn *redundant representations* — multiple neurons capture similar features. This makes the network robust.

3. **Regularisation view:** Dropout adds noise to the hidden representations, which acts as a form of data augmentation at the feature level. This prevents co-adaptation — neurons can't conspire to memorise specific training examples.

## 4.3 Practical details

| Detail | Value |
|--------|-------|
| Typical drop rate | 0.2–0.5 (0.2 is a good default) |
| Where to apply | After activation (ReLU), before next linear layer |
| Training mode | `model.train()` — dropout is active |
| Eval mode | `model.eval()` — dropout is OFF (use all neurons) |
| Input dropout | Rarely used (maybe 0.1 for very noisy inputs) |
| Last layer | Never apply dropout after the final layer |

In [ ]:
# Dropout from scratch — understand exactly what PyTorch does

def dropout_forward(h, p=0.5, training=True):
    """
    Inverted dropout.
    h: activations (any shape)
    p: probability of DROPPING each neuron
    """
    if not training or p == 0:
        return h  # at test time, return unchanged
    
    # Generate binary mask: 1 = keep, 0 = drop
    mask = (np.random.rand(*h.shape) > p).astype(np.float32)
    
    # Apply mask AND scale up survivors
    return h * mask / (1 - p)
    #     ^^^^^^^^      ^^^^^^^^
    #     zero some     scale up survivors so
    #     neurons       expected value is preserved

# Demo
np.random.seed(42)
h = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])

print("Original activations:", h)
print(f"Mean: {h.mean():.2f}")

# Apply dropout 5 times to show randomness
print("\nDropout (p=0.5) applied 5 times:")
for i in range(5):
    dropped = dropout_forward(h, p=0.5)
    n_zeros = (dropped == 0).sum()
    print(f"  Trial {i+1}: {dropped.round(1)} | {n_zeros}/10 dropped | mean={dropped.mean():.2f}")

# Show that expected value is preserved
print("\nExpected value test (1000 trials):")
means = [dropout_forward(h, p=0.5).mean() for _ in range(1000)]
print(f"  Original mean: {h.mean():.2f}")
print(f"  Avg dropped mean: {np.mean(means):.2f}  (should be ≈ {h.mean():.2f})")
print(f"  ✓ Inverted dropout preserves the expected value.")

In [ ]:
# Visualise dropout effect on activations
np.random.seed(42)
h = np.random.randn(1, 20)  # 20 neurons

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

configs = [
    ("No dropout (p=0)", 0.0, "#378ADD"),
    ("Light dropout (p=0.2)", 0.2, "#1D9E75"),
    ("Standard dropout (p=0.5)", 0.5, "#7F77DD"),
    ("Heavy dropout (p=0.8)", 0.8, "#E24B4A"),
]

for ax, (title, p, color) in zip(axes, configs):
    dropped = dropout_forward(h.copy(), p=p)
    bar_colors = [color if v != 0 else '#ddd' for v in dropped[0]]
    ax.bar(range(20), dropped[0], color=bar_colors)
    n_active = (dropped[0] != 0).sum()
    ax.set_title(f'{title}\n{n_active}/20 active', fontweight='bold', fontsize=10)
    ax.set_ylim(-3, 4)
    ax.set_ylabel('Activation')
    ax.axhline(0, color='gray', linewidth=0.5)

plt.suptitle('Dropout: higher p → more neurons zeroed out, survivors scaled up',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 4.4 Dropout on MNIST — does it help?

In [ ]:
# Compare: no dropout vs dropout=0.2 vs dropout=0.5

def make_model_with_dropout(p=0.0):
    torch.manual_seed(42)
    layers = [
        nn.Flatten(),
        nn.Linear(784, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
    ]
    if p > 0:
        layers.append(nn.Dropout(p))
    layers += [
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
    ]
    if p > 0:
        layers.append(nn.Dropout(p))
    layers.append(nn.Linear(128, 10))
    return nn.Sequential(*layers).to(device)

dropout_configs = {
    "No dropout": 0.0,
    "Dropout=0.2": 0.2,
    "Dropout=0.5": 0.5,
}

dropout_results = {}
N_EPOCHS_DROP = 15

for name, p in dropout_configs.items():
    print(f"\nTraining: {name}")
    model = make_model_with_dropout(p)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, N_EPOCHS_DROP + 1):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, test_loader, criterion)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        if epoch % 5 == 0:
            print(f"  Epoch {epoch:2d} | Train: {ta:.2%} | Val: {va:.2%} | Gap: {ta-va:+.2%}")
    
    dropout_results[name] = history
    print(f"  Best val: {max(history['val_acc']):.2%} | Train-val gap: {history['train_acc'][-1]-history['val_acc'][-1]:.2%}")

In [ ]:
# === DROPOUT COMPARISON PLOT ===
drop_colors = {'No dropout': '#E24B4A', 'Dropout=0.2': '#1D9E75', 'Dropout=0.5': '#7F77DD'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Validation accuracy
for name, hist in dropout_results.items():
    c = drop_colors[name]
    axes[0].plot(range(1, N_EPOCHS_DROP+1), hist['val_acc'], 'o-', color=c,
                linewidth=2, markersize=3, label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation accuracy')
axes[0].set_title('Validation accuracy', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

# 2. Train vs val loss (overfitting check)
for name, hist in dropout_results.items():
    c = drop_colors[name]
    axes[1].plot(range(1, N_EPOCHS_DROP+1), hist['train_loss'], '-', color=c,
                linewidth=2, alpha=0.5, label=f'{name} (train)')
    axes[1].plot(range(1, N_EPOCHS_DROP+1), hist['val_loss'], '--', color=c,
                linewidth=2, label=f'{name} (val)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Train vs val loss (solid=train, dashed=val)', fontweight='bold')
axes[1].legend(fontsize=7, ncol=2); axes[1].grid(True, alpha=0.2)

# 3. Generalisation gap (train_acc - val_acc)
for name, hist in dropout_results.items():
    c = drop_colors[name]
    gap = [t - v for t, v in zip(hist['train_acc'], hist['val_acc'])]
    axes[2].plot(range(1, N_EPOCHS_DROP+1), gap, 'o-', color=c,
                linewidth=2, markersize=3, label=name)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Train acc − Val acc')
axes[2].set_title('Generalisation gap (lower = better)', fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.2)
axes[2].axhline(0, color='gray', linewidth=0.5)

plt.suptitle('Dropout ablation on MNIST', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("• No dropout: highest train acc but LARGEST train-val gap (overfitting)")
print("• Dropout=0.2: slightly lower train acc but SMALLER gap (better generalisation)")
print("• Dropout=0.5: may hurt if the model is small (not enough capacity left)")
print("\nFor MNIST, dropout=0.2 is usually the sweet spot.")
print("For larger models (ResNets, transformers), dropout=0.1-0.3 is common.")

---
# Part 5 — Practical Guidelines

## When to use what

| Scenario | Optimiser | Dropout | Why |
|----------|-----------|---------|-----|
| Starting a new project | **Adam (lr=1e-3)** | 0.1–0.2 | Fast convergence, good default |
| Fine-tuning a pretrained model | **Adam (lr=1e-4)** or **SGD (lr=1e-3, mom=0.9)** | 0.1 | Lower lr to preserve pretrained features |
| Training from scratch on large data | **SGD (lr=0.1, mom=0.9) + cosine schedule** | 0.0–0.1 | Often better final accuracy than Adam |
| Small dataset, overfitting | Adam | **0.3–0.5** | Need strong regularisation |
| Very deep network (50+ layers) | Adam | **0.1** | Dropout + batch norm together |
| Transformers / attention models | **AdamW** (Adam with decoupled weight decay) | 0.1 | AdamW handles L2 reg correctly |

## Common mistakes

1. **Using the same lr for SGD and Adam.** Adam needs ~10× smaller lr (1e-3 vs 1e-2 for SGD)
2. **Forgetting `model.eval()`** before validation → dropout still active → noisy results
3. **Dropout too high on small models** → not enough capacity left → underfitting
4. **Not using any regularisation** → overfitting, especially on small datasets
5. **Applying dropout after the last layer** → you're dropping output neurons!

## The typical modern recipe

```python
model = MyModel()  # with BatchNorm + Dropout(0.1-0.2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
```

This combination of Adam + weight decay (L2) + dropout + batch norm + cosine LR schedule is the standard starting point for most projects.

---
## Summary

| Optimiser | State | Key idea | When |
|-----------|-------|----------|------|
| **SGD** | None | Raw gradient | Baseline / research |
| **Momentum** | velocity $v$ | Exponential average of gradients | Better than raw SGD |
| **RMSProp** | squared grad $s$ | Adaptive per-parameter lr | When scales vary |
| **Adam** | $m$ + $v$ + bias | Momentum + RMSProp | **Default choice** |

| Dropout | What | When |
|---------|------|------|
| p=0 | No dropout | Very large datasets, strong data augmentation |
| p=0.1–0.2 | Light | **Default for most models** |
| p=0.3–0.5 | Heavy | Small datasets, overfitting |

**The key connection:** Optimisers control *how fast and where* we move in weight space. Dropout controls *how complex* a solution the network can find. Together they determine whether your model converges quickly to a solution that generalises.